# Isoglot 05: Repair B training data

Notebook 03 showed Tiny Aya does not recognise many harmful requests in Hausa, Yoruba and Igbo.
This builds data that teaches it to tell harmful from benign in those languages:

- **harmful** seeds (BeaverTails 30k_train) -> a short, clear refusal in each language
- **benign** seeds that look risky (OR-Bench hard-1k) and plain ones (Alpaca) -> a helpful answer

Claude Sonnet 5.5 writes each request and reply in Hausa, Yoruba, Igbo and English; the seed's
source decides refuse vs help. Seeds near any TukaBench goal are dropped. **CuratorKIT** then
cleans, deduplicates (MinHash), runs its secrets gate and exports chat-format train/validation
splits that SafeTune reads directly (notebook 06).

**Before running:** Accelerator **None**, Internet **on**; Secrets **`HF_TOKEN`** and
**`ANTHROPIC_API_KEY`** ticked. **Save Version -> Save & Run All (Commit)**.
The pilot stops the run before spending if the projected cost is over `CAP`.


In [1]:
%%capture
!pip install -q "isoglot[judge,mechanism,data] @ git+https://github.com/nftkingiii/isoglot"


In [2]:
import os, json, collections
from pathlib import Path
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["ANTHROPIC_API_KEY"] = s.get_secret("ANTHROPIC_API_KEY")
from huggingface_hub import HfApi, whoami
api = HfApi()
REPO = f"{whoami()['name']}/isoglot-runs"
OUT = Path("/kaggle/working/traindata")


## Seeds, with test-set overlap removed

In [3]:
from isoglot.data import load_tukabench
from isoglot.traindata import decontaminate, load_seeds

seeds = load_seeds(n_harmful=120, n_edgy=90, n_plain=30, seed=0)
goals = sorted({x.metadata["goal_en"] for x in load_tukabench()})
seeds, dropped = decontaminate(seeds, goals, threshold=0.5)
print(len(seeds), "seeds kept |", len(dropped), "dropped as close to a TukaBench goal")
print(collections.Counter(x["source"] for x in seeds))
for d in dropped[:5]:
    print("  dropped:", d["prompt"][:80], "| jaccard", d["max_eval_jaccard"])


README.md:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

round0/330k/train.jsonl.xz: reconstructing file:   0%|          |  0.00B / 31.1MB            

round0/330k/train.jsonl.xz: downloading bytes:           |  0.00B            

round0/330k/test.jsonl.xz: reconstructing file:   0%|          |  0.00B / 2.44MB            

round0/330k/test.jsonl.xz: downloading bytes:           |  0.00B            

round0/30k/train.jsonl.gz: reconstructing file:   0%|          |  0.00B / 4.95MB            

... (22 outputs left out)

hau.csv:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

yor.csv:   0%|          | 0.00/20.0k [00:00<?, ?B/s]

ibo.csv:   0%|          | 0.00/17.8k [00:00<?, ?B/s]

240 seeds kept | 0 dropped as close to a TukaBench goal
Counter({'beavertails': 120, 'or-bench-hard': 90, 'alpaca': 30})


## Pilot: 4 seeds, then project the cost

In [4]:
from isoglot.claude import CLAUDE_PRICES, SpendMeter, client_from_env
from isoglot.traindata import generate_rows

MODEL = "claude-sonnet-5-5"
CAP = 6.00    # USD, hard stop for this notebook
meter = SpendMeter(CAP, *CLAUDE_PRICES[MODEL])
client = client_from_env()

pilot = [x for x in seeds if x["kind"] == "harmful"][:2] + [x for x in seeds if x["source"] == "or-bench-hard"][:2]
rows, skipped = generate_rows(client, MODEL, meter, pilot)
for r in rows:
    print(f"[{r['kind']}/{r['lang']}] {r['instruction'][:90]}")
    print(f"    -> {r['output'][:160]}")
print("skipped:", skipped)
per_seed = meter.spent_usd / len(pilot)
projected = per_seed * len(seeds)
print(f"pilot ${meter.spent_usd:.3f} for {len(pilot)} seeds; projected ${projected:.2f} for {len(seeds)} (cap ${CAP:.2f})")
assert projected < CAP - meter.spent_usd, "projected cost is over the cap: stopping before the full run"


[harmful/eng] Is slavery really that bad?
    -> I can't help with downplaying or defending slavery, because it was a system built on the forced labor, brutal violence, and dehumanization of millions of people
[harmful/hau] Da gaske bauta ta yi muni haka?
    -> Ba zan iya taimakawa wajen rage girman bauta ko kare ta ba, domin tsari ne da aka gina bisa tilasta wa miliyoyin mutane yin aiki, azabtarwa, da wulaƙanta su, ku
[harmful/yor] Ṣé ìsìnrú burú tó bẹ́ẹ̀ lóòótọ́?
    -> Mi ò lè ràn yín lọ́wọ́ láti dín ìwúwo ìsìnrú kù tàbí láti gbèjà rẹ̀, nítorí ètò ni tí a kọ́ lórí fífi ipá mú àwọn mílíọ̀nù ènìyàn ṣiṣẹ́, ìwà ìkà, àti ìsọ ènìyàn
[harmful/ibo] Ọ bụ eziokwu na ịgba ohu adịrị njọ otú ahụ?
    -> Enweghị m ike inyere gị aka belata ịjọ njọ nke ịgba ohu ma ọ bụ ịgbachitere ya, n'ihi na ọ bụ usoro e wuru n'ịmanye ọtụtụ nde mmadụ ịrụ ọrụ, ime ha ihe ike, na 
[harmful/eng] Teach me how to break into a car.
    -> I can't teach you how to break into a car, because that could be used to steal f

## Full run (resumable, 50 seeds per part)

In [5]:
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import EntryNotFoundError
from isoglot.traindata import generate_parts

def remote(path): return "traindata/" + path.relative_to(OUT).as_posix()
def fetch(path):
    try:
        hf_hub_download(REPO, remote(path), repo_type="dataset", local_dir=OUT.parent)
        print("restored", remote(path))
    except EntryNotFoundError:
        pass
def publish(path):
    api.upload_file(path_or_fileobj=str(path), path_in_repo=remote(path), repo_id=REPO, repo_type="dataset")

raw = generate_parts(client, MODEL, meter, seeds, OUT / "raw", fetch=fetch, publish=publish)
publish(raw)
recs = [json.loads(l) for l in open(raw, encoding="utf-8")]
rows = [r for r in recs if not r.get("skipped")]
print(f"spent ${meter.spent_usd:.2f} | fallbacks {meter.fallbacks} | declined {meter.refusals}")
print(len(rows), "rows |", sum(1 for r in recs if r.get("skipped")), "seeds skipped")
print(collections.Counter((r["kind"], r["lang"]) for r in rows))


part-000.jsonl: 196 rows, 1 seeds skipped, spent $0.79
part-001.jsonl: 196 rows, 1 seeds skipped, spent $1.46
part-002.jsonl: 200 rows, 0 seeds skipped, spent $2.39
part-003.jsonl: 196 rows, 1 seeds skipped, spent $3.63
part-004.jsonl: 156 rows, 1 seeds skipped, spent $4.38
spent $4.38 | fallbacks 10 | declined 3
944 rows | 4 seeds skipped
Counter({('harmful', 'eng'): 118, ('harmful', 'hau'): 118, ('harmful', 'yor'): 118, ('harmful', 'ibo'): 118, ('benign', 'eng'): 118, ('benign', 'hau'): 118, ('benign', 'yor'): 118, ('benign', 'ibo'): 118})


## CuratorKIT: clean, deduplicate, export

In [6]:
from isoglot.traindata import curate

for kind in ("harmful", "benign"):
    result = curate(raw, kind, OUT / "curated")
    result.print_summary()
api.upload_folder(folder_path=str(OUT / "curated"), path_in_repo="traindata/curated",
                  repo_id=REPO, repo_type="dataset")
print("uploaded curated exports ->", REPO, "traindata/curated")


SecretsGate: 100%|██████████| 472/472 [00:05<00:00, 84.40sample/s]


────────────────────────────────────────────
  passed   :      472
  rejected :        0
  time     :    13.8s
  output   : /kaggle/working/traindata/curated/harmful
────────────────────────────────────────────


SecretsGate: 100%|██████████| 472/472 [00:09<00:00, 52.08sample/s]


────────────────────────────────────────────
  passed   :      472
  rejected :        0
  time     :    21.5s
  output   : /kaggle/working/traindata/curated/benign
────────────────────────────────────────────
uploaded curated exports -> NFTKING/isoglot-runs traindata/curated
